In [3]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
    )
    model.fit(
        train_df[feat_cols].values,
        np.log1p(train_df["target"].values),
        eval_set=(val_df[feat_cols].values, np.log1p(val_df["target"].values)),
    )
    preds = np.expm1(model.predict(val_df[feat_cols].values))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration_


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val1_df = load_split("split2", feature_name)
    val2_df = load_split("split3", feature_name)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": "split1_" + feature_name, "groups": feature_name, "rmsle": score1})
    results.append({"experiment": "split2_" + feature_name, "groups": feature_name, "rmsle": score2})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val1_df = merge_splits("split2", subset)
    val2_df = merge_splits("split3", subset)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": f"split1_cumulative_{i}", "groups": label, "rmsle": score1})
    results.append({"experiment": f"split2_cumulative_{i}", "groups": label, "rmsle": score2})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))

 split1 RMSLE = 1.74429  (best iteration: 584) | split2 RMSLE = 1.70080  (best iteration: 220)

Ranking by RMSLE (lower = better)
             experiment                                                                                 groups    rmsle
           split2_trend                                                                                  trend 1.697419
        split2_velocity                                                                               velocity 1.697837
 split2_rolling_windows                                                                        rolling_windows 1.697849
    split2_cumulative_6                          base + all_history + rolling_windows + velocity + trend + gmv 1.697936
    split2_cumulative_7       base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.698992
    split2_cumulative_8 base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.700802
    split2_cumulative_5       

In [4]:
import joblib
import catboost as cb
import optuna

RANDOM_STATE = 42

cb_train = pd.read_parquet("datasets/split1_trend.parquet")
cb_val = pd.read_parquet("datasets/split2_trend.parquet")
cb_val2 = pd.read_parquet("datasets/split3_trend.parquet")

cb_feat_cols = [c for c in cb_train.columns if
                c not in ("user_id", "target") and not pd.api.types.is_datetime64_any_dtype(cb_train[c])]

print(f"  CatBoost  — train: {cb_train.shape[0]} rows, {len(cb_feat_cols)} features")

X_cb_tr = cb_train[cb_feat_cols].values
y_cb_tr = np.log1p(cb_train["target"].values)
X_cb_val = cb_val[cb_feat_cols].values
y_cb_val = np.log1p(cb_val["target"].values)
X_cb_val2 = cb_val2[cb_feat_cols].values
y_cb_val2 = np.log1p(cb_val2["target"].values)
y_cb_val_raw = cb_val["target"].values
y_cb_val2_raw = cb_val2["target"].values

print("\nOptuna tuning CatBoost (20 trials)...")


def cb_objective(trial):
    params = dict(
        loss_function="RMSE",
        iterations=trial.suggest_int("iterations", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
        random_seed=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=50,
    )
    model = cb.CatBoostRegressor(**params)
    model.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))
    score1 = rmsle(y_cb_val_raw, np.expm1(model.predict(X_cb_val)))
    score2 = rmsle(y_cb_val2_raw, np.expm1(model.predict(X_cb_val2)))
    return (score1 + score2) / 2


cb_study = optuna.create_study(direction="minimize")
cb_study.optimize(cb_objective, n_trials=20)
best_cb_params = cb_study.best_params

print(f"  Best CatBoost RMSLE: {cb_study.best_value:.5f}")
print(f"  Params: {best_cb_params}")

final_cb = cb.CatBoostRegressor(
    **best_cb_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    early_stopping_rounds=50,
)
final_cb.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))

cb_best_n = final_cb.best_iteration_ + 1

X_cb_full = np.concatenate([X_cb_tr, X_cb_val])
y_cb_full = np.concatenate([y_cb_tr, y_cb_val])

cb_sub_params = {k: v for k, v in best_cb_params.items() if k not in ["iterations", "early_stopping_rounds"]}

final_cb_full = cb.CatBoostRegressor(
    **cb_sub_params,
    iterations=cb_best_n,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
final_cb_full.fit(X_cb_full, y_cb_full)

joblib.dump(final_cb, "model/cat_split1.joblib")
joblib.dump(final_cb_full, "model/cat_full.joblib")

['model/cat_full.joblib']

In [4]:
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(feature_name: str, t: str) -> pd.DataFrame:
    path = f"datasets/{t}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(t: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(feature_names[0], t)
    for name in feature_names[1:]:
        extra = load_split(name, t)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
    )
    model.fit(
        train_df[feat_cols].values,
        np.log1p(train_df["target"].values),
        eval_set=(val_df[feat_cols].values, np.log1p(val_df["target"].values)),
    )
    preds = np.expm1(model.predict(val_df[feat_cols].values))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration_


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split(feature_name, "train")
    val_df = load_split(feature_name, "test")
    score, best_iter = train_and_eval(train_df, val_df)
    print(
        f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": feature_name, "rmsle": score})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("train", subset)
    val_df = merge_splits("test", subset)
    score, best_iter = train_and_eval(train_df, val_df)
    print(
        f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": label, "rmsle": score})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))

 RMSLE = 1.66969  (best iteration: 793)

Ranking by RMSLE (lower = better)
                                                                            experiment    rmsle
                         base + all_history + rolling_windows + velocity + trend + gmv 1.669251
                               base + all_history + rolling_windows + velocity + trend 1.669370
base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.669686
                                                  base + all_history + rolling_windows 1.669716
      base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.669728
                                       base + all_history + rolling_windows + velocity 1.669933
                                                                                 trend 1.670119
                                                                       rolling_windows 1.670554
                                                             

In [21]:
import joblib
import catboost as cb
import optuna

RANDOM_STATE = 42

cb_train = pd.read_parquet("datasets/train_all_features.parquet")
cb_val = pd.read_parquet("datasets/train_all_features.parquet")

cb_feat_cols = [c for c in cb_train.columns if
                c not in ("user_id", "target") and not pd.api.types.is_datetime64_any_dtype(cb_train[c])]

print(f"  CatBoost  — train: {cb_train.shape[0]} rows, {len(cb_feat_cols)} features")

X_cb_tr = cb_train[cb_feat_cols].values
y_cb_tr = np.log1p(cb_train["target"].values)
X_cb_val = cb_val[cb_feat_cols].values
y_cb_val = np.log1p(cb_val["target"].values)
y_cb_val_raw = cb_val["target"].values

print("\nOptuna tuning CatBoost (20 trials)...")


def cb_objective(trial):
    params = dict(
        loss_function="RMSE",
        iterations=trial.suggest_int("iterations", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
        random_seed=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=50,
    )
    model = cb.CatBoostRegressor(**params)
    model.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))
    score = rmsle(y_cb_val_raw, np.expm1(model.predict(X_cb_val)))
    return score


cb_study = optuna.create_study(direction="minimize")
cb_study.optimize(cb_objective, n_trials=20)
best_cb_params = cb_study.best_params

print(f"  Best CatBoost RMSLE: {cb_study.best_value:.5f}")
print(f"  Params: {best_cb_params}")

final_cb = cb.CatBoostRegressor(
    **best_cb_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    early_stopping_rounds=50,
)
final_cb.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))

cb_best_n = final_cb.best_iteration_ + 1

X_cb_full = np.concatenate([X_cb_tr, X_cb_val])
y_cb_full = np.concatenate([y_cb_tr, y_cb_val])

cb_sub_params = {k: v for k, v in best_cb_params.items() if k not in ["iterations", "early_stopping_rounds"]}

final_cb_full = cb.CatBoostRegressor(
    **cb_sub_params,
    iterations=cb_best_n,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
final_cb_full.fit(X_cb_full, y_cb_full)

joblib.dump(final_cb, "model/cat_split1_all_features.joblib")
joblib.dump(final_cb_full, "model/cat_full_all_features.joblib")

['model/cat_full_all_features.joblib']

In [6]:
import pandas as pd
import numpy as np
import joblib
import catboost as cb
import optuna

RANDOM_STATE = 42

# Загружаем три сплита и конкатенируем
frames = []
for split_name in ["split1", "split2", "split3"]:
  frame = pd.read_parquet(f"datasets/{split_name}_all_history_rolling_windows_velocity_trend_gmv.parquet")
  frame["split"] = split_name
  frames.append(frame)

cb_full = pd.concat(frames, ignore_index=True)

test_df = pd.read_parquet("datasets/test_all_features.parquet")
test_users = set(test_df["user_id"].unique())

cb_train = cb_full[~cb_full["user_id"].isin(test_users)]
cb_val   = cb_full[cb_full["user_id"].isin(test_users)]

print(f"  Train: {cb_train.shape[0]} rows, {cb_train['user_id'].nunique()} users")
print(f"  Val:   {cb_val.shape[0]} rows, {cb_val['user_id'].nunique()} users")

cb_feat_cols = [c for c in cb_train.columns if
                c not in ("user_id", "target", "split") and not pd.api.types.is_datetime64_any_dtype(cb_train[c])]

print(f"  CatBoost  — train: {cb_train.shape[0]} rows, {len(cb_feat_cols)} features")

X_cb_tr = cb_train[cb_feat_cols].values
y_cb_tr = np.log1p(cb_train["target"].values)
X_cb_val = cb_val[cb_feat_cols].values
y_cb_val = np.log1p(cb_val["target"].values)
y_cb_val_raw = cb_val["target"].values

print("\nOptuna tuning CatBoost (20 trials)...")


def cb_objective(trial):
    params = dict(
        loss_function="RMSE",
        iterations=trial.suggest_int("iterations", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
        random_seed=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=50,
    )
    model = cb.CatBoostRegressor(**params)
    model.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))
    score = rmsle(y_cb_val_raw, np.expm1(model.predict(X_cb_val)))
    return score


cb_study = optuna.create_study(direction="minimize")
cb_study.optimize(cb_objective, n_trials=20)
best_cb_params = cb_study.best_params

print(f"  Best CatBoost RMSLE: {cb_study.best_value:.5f}")
print(f"  Params: {best_cb_params}")

final_cb = cb.CatBoostRegressor(
    **best_cb_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    early_stopping_rounds=50,
)
final_cb.fit(X_cb_tr, y_cb_tr, eval_set=(X_cb_val, y_cb_val))

cb_best_n = final_cb.best_iteration_ + 1

X_cb_full = np.concatenate([X_cb_tr, X_cb_val])
y_cb_full = np.concatenate([y_cb_tr, y_cb_val])

cb_sub_params = {k: v for k, v in best_cb_params.items() if k not in ["iterations", "early_stopping_rounds"]}

final_cb_full = cb.CatBoostRegressor(
    **cb_sub_params,
    iterations=cb_best_n,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
final_cb_full.fit(X_cb_full, y_cb_full)

joblib.dump(final_cb, "model/cat_split1_2_3_concat_all_features.joblib")
joblib.dump(final_cb_full, "model/cat_full_all_features.joblib")

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=108]="split1": Cannot convert 'split1' to float